# 🛡️ Passive Sonar Threat Classification — Quantized Transformer on FPGA
**Domain:** Submarine Warfare | Passive Sonar | Edge AI

**Pipeline:**
```
Raw Audio → STFT → Mel Spectrogram → Patch Tokens → Quantized ViT → Class + FPGA Estimate
```

**Classes (DeepShip → Defense Mapping):**
| Dataset Label | Defense Label |
|---|---|
| Cargo | Non-Threat |
| Tanker | Non-Threat |
| Passengership | Monitor |
| Tug | Potential Hostile |

## Phase 1 — Environment Setup

In [ ]:
# Install all dependencies
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install librosa gdown numpy matplotlib scikit-learn seaborn tqdm einops

In [ ]:
import os, math, time, random, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
from pathlib import Path
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.quantization import quantize_dynamic
from sklearn.metrics import classification_report, confusion_matrix
from einops import rearrange
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Phase 2 — Dataset Download (DeepShip)

In [ ]:
# ── DeepShip Download Instructions ──────────────────────────────────────────
# DeepShip is hosted on IEEE DataPort. Two ways to get it:
#
# OPTION A — Direct (if you have IEEE account):
#   https://ieee-dataport.org/open-access/deepship
#   Download and extract to ./data/DeepShip/
#
# OPTION B — GitHub mirror (smaller, pre-processed):
#   git clone https://github.com/irfankamboh/DeepShip
#   Copy the audio folders to ./data/DeepShip/
#
# Expected folder structure after extraction:
# data/DeepShip/
#   ├── Cargo/        (*.wav files)
#   ├── Passengership/(*.wav files)
#   ├── Tanker/       (*.wav files)
#   └── Tug/          (*.wav files)
# ─────────────────────────────────────────────────────────────────────────────

DATA_ROOT = Path('./data/DeepShip')
CLASSES = ['Cargo', 'Passengership', 'Tanker', 'Tug']
DEFENSE_MAP = {
    'Cargo': 'Non-Threat',
    'Passengership': 'Monitor',
    'Tanker': 'Non-Threat',
    'Tug': 'Potential-Hostile'
}
CLASS2IDX = {c: i for i, c in enumerate(CLASSES)}

# Verify structure
if DATA_ROOT.exists():
    for cls in CLASSES:
        files = list((DATA_ROOT / cls).glob('*.wav'))
        print(f'  {cls:20s}: {len(files):4d} files')
else:
    print('⚠  data/DeepShip not found. Follow OPTION A or B above first.')

## Phase 3 — Signal Processing Pipeline
### STFT → Mel Spectrogram → Patch Tokenization

In [ ]:
# ── Audio Config ─────────────────────────────────────────────────────────────
SAMPLE_RATE   = 22050   # Hz
CLIP_DURATION = 5       # seconds per sample
N_MELS        = 128     # mel filterbanks
N_FFT         = 1024    # FFT window size
HOP_LENGTH    = 512     # STFT hop
IMG_H         = 128     # spectrogram height (n_mels)
IMG_W         = 216     # spectrogram width  (~5s at sr=22050, hop=512)
PATCH_SIZE    = 16      # patch side length (pixels)
NUM_PATCHES   = (IMG_H // PATCH_SIZE) * (IMG_W // PATCH_SIZE)  # = 8*13 = 104

print(f'Spectrogram shape : ({IMG_H}, {IMG_W})')
print(f'Patch grid        : {IMG_H//PATCH_SIZE} x {IMG_W//PATCH_SIZE}')
print(f'Total patches (tokens): {NUM_PATCHES}')

def load_and_process(filepath, augment=False):
    """Load WAV → Mel Spectrogram (log scale, normalized)"""
    y, sr = librosa.load(filepath, sr=SAMPLE_RATE,
                          duration=CLIP_DURATION, mono=True)

    # Pad/trim to exact length
    target_len = SAMPLE_RATE * CLIP_DURATION
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]

    # Optional: time-shift augmentation
    if augment:
        shift = random.randint(-2000, 2000)
        y = np.roll(y, shift)

    # Mel spectrogram
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)  # log scale

    # Resize to fixed IMG_H x IMG_W
    if log_mel.shape[1] != IMG_W:
        log_mel = librosa.util.fix_length(log_mel, size=IMG_W, axis=1)

    # Normalize to [0, 1]
    log_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-8)

    return log_mel.astype(np.float32)  # shape: (128, 216)

In [ ]:
# ── Visualize one sample per class ───────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('Mel Spectrograms — Passive Sonar Signatures', fontsize=14, fontweight='bold')

for i, cls in enumerate(CLASSES):
    files = list((DATA_ROOT / cls).glob('*.wav'))
    if not files:
        continue
    mel = load_and_process(files[0])
    im = axes[i].imshow(mel, aspect='auto', origin='lower',
                         cmap='viridis', vmin=0, vmax=1)
    axes[i].set_title(f'{cls}\n({DEFENSE_MAP[cls]})', fontsize=11)
    axes[i].set_xlabel('Time frames')
    axes[i].set_ylabel('Mel bins')

plt.tight_layout()
plt.savefig('./sonar_spectrograms.png', dpi=150)
plt.show()
print('Saved: sonar_spectrograms.png')

## Phase 4 — Dataset Class & DataLoaders

In [ ]:
class DeepShipDataset(Dataset):
    def __init__(self, root, classes, split='train',
                 split_ratio=(0.7, 0.15, 0.15), augment=False):
        self.samples = []
        self.augment = augment

        all_files = []
        for cls in classes:
            files = sorted((root / cls).glob('*.wav'))
            all_files.extend([(f, CLASS2IDX[cls]) for f in files])

        random.shuffle(all_files)
        n = len(all_files)
        tr = int(n * split_ratio[0])
        vl = int(n * split_ratio[1])

        if split == 'train':
            self.samples = all_files[:tr]
        elif split == 'val':
            self.samples = all_files[tr:tr+vl]
        else:
            self.samples = all_files[tr+vl:]

        print(f'  {split:5s}: {len(self.samples)} samples')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        mel = load_and_process(path, augment=self.augment)
        # shape → (1, H, W) for single-channel input
        mel = torch.tensor(mel).unsqueeze(0)
        return mel, label


print('Building datasets...')
train_ds = DeepShipDataset(DATA_ROOT, CLASSES, split='train', augment=True)
val_ds   = DeepShipDataset(DATA_ROOT, CLASSES, split='val')
test_ds  = DeepShipDataset(DATA_ROOT, CLASSES, split='test')

BATCH = 32
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=4, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=4, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=4, pin_memory=True)

print(f'\nBatch shape: {next(iter(train_dl))[0].shape}')

## Phase 5 — Quantized Vision Transformer
### Core Contribution: Dynamic-Scale Attention Softmax

In [ ]:
# ── Patch Embedding ───────────────────────────────────────────────────────────
class PatchEmbedding(nn.Module):
    """Splits spectrogram into patches and linearly projects to embed_dim."""
    def __init__(self, img_h=IMG_H, img_w=IMG_W,
                 patch_size=PATCH_SIZE, in_channels=1, embed_dim=256):
        super().__init__()
        self.n_patches = (img_h // patch_size) * (img_w // patch_size)
        self.proj = nn.Conv2d(in_channels, embed_dim,
                              kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        # x: (B, 1, H, W) → (B, embed_dim, H/P, W/P) → (B, n_patches, embed_dim)
        x = self.proj(x)                        # (B, D, h, w)
        x = x.flatten(2).transpose(1, 2)        # (B, n_patches, D)
        return x


# ── Dynamic-Scale Multi-Head Attention (Core Contribution) ────────────────────
class DynamicScaleAttention(nn.Module):
    """
    Standard MHA + dynamic range calibration before softmax.

    Problem it solves:
      In sonar/radar signals, QKᵀ dynamic range shifts wildly under
      jamming or ocean noise floor changes. Static INT8 scale → attention
      collapse. This module tracks per-step max(|QKᵀ|) and applies an
      adaptive scale before softmax — hardware-equivalent logic.
    """
    def __init__(self, embed_dim, num_heads, dropout=0.1, use_dynamic_scale=True):
        super().__init__()
        self.num_heads   = num_heads
        self.head_dim    = embed_dim // num_heads
        self.scale       = self.head_dim ** -0.5
        self.use_dynamic = use_dynamic_scale

        self.qkv     = nn.Linear(embed_dim, 3 * embed_dim)
        self.proj    = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

        # Learnable scale correction factor (simulates hardware register)
        self.scale_correction = nn.Parameter(torch.ones(1))

    def forward(self, x):
        B, N, D = x.shape
        H = self.num_heads

        qkv = self.qkv(x).reshape(B, N, 3, H, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)             # each: (B, H, N, head_dim)

        attn = (q @ k.transpose(-2, -1))    # (B, H, N, N) — raw QKᵀ

        if self.use_dynamic:
            # ── DYNAMIC SCALE UNIT ──────────────────────────────────────────
            # Simulates hardware: compute max(|QKᵀ|) per inference step,
            # adjust quantization scale register before softmax.
            # In FPGA: this is a max-reduce tree + scale register update
            # running parallel to the attention pipeline.
            dynamic_range = attn.abs().amax(dim=(-2, -1), keepdim=True)  # (B,H,1,1)
            dynamic_range = dynamic_range.clamp(min=1.0)  # avoid div by zero
            attn_normalized = attn / dynamic_range         # bring into safe range
            attn_scaled     = attn_normalized * self.scale * self.scale_correction
            # ────────────────────────────────────────────────────────────────
        else:
            attn_scaled = attn * self.scale

        attn_weights = F.softmax(attn_scaled, dim=-1)
        attn_weights = self.dropout(attn_weights)

        out = (attn_weights @ v).transpose(1, 2).reshape(B, N, D)
        return self.proj(out), attn_weights


# ── Transformer Block ─────────────────────────────────────────────────────────
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0,
                 dropout=0.1, use_dynamic_scale=True):
        super().__init__()
        self.norm1  = nn.LayerNorm(embed_dim)
        self.attn   = DynamicScaleAttention(embed_dim, num_heads,
                                            dropout, use_dynamic_scale)
        self.norm2  = nn.LayerNorm(embed_dim)
        mlp_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        attn_out, weights = self.attn(self.norm1(x))
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x, weights


# ── Full Sonar Transformer ────────────────────────────────────────────────────
class SonarTransformer(nn.Module):
    def __init__(self,
                 img_h=IMG_H, img_w=IMG_W,
                 patch_size=PATCH_SIZE,
                 in_channels=1,
                 num_classes=len(CLASSES),
                 embed_dim=256,
                 depth=6,
                 num_heads=8,
                 mlp_ratio=4.0,
                 dropout=0.1,
                 use_dynamic_scale=True):
        super().__init__()

        self.patch_embed = PatchEmbedding(img_h, img_w, patch_size,
                                          in_channels, embed_dim)
        n_patches = self.patch_embed.n_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, n_patches + 1, embed_dim) * 0.02)
        self.pos_drop  = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, dropout, use_dynamic_scale)
            for _ in range(depth)
        ])

        self.norm       = nn.LayerNorm(embed_dim)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim // 2, num_classes)
        )

        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x, return_attn=False):
        B = x.shape[0]
        x = self.patch_embed(x)                              # (B, N, D)

        cls = self.cls_token.expand(B, -1, -1)               # (B, 1, D)
        x   = torch.cat([cls, x], dim=1)                     # (B, N+1, D)
        x   = self.pos_drop(x + self.pos_embed)

        attn_maps = []
        for block in self.blocks:
            x, w = block(x)
            attn_maps.append(w)

        x   = self.norm(x)
        cls = x[:, 0]                                        # CLS token
        out = self.classifier(cls)

        if return_attn:
            return out, attn_maps
        return out


# Build model
model = SonarTransformer(use_dynamic_scale=True).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters  : {total_params:,}')
print(f'Trainable params  : {trainable:,}')
print(f'Model size (FP32) : {total_params * 4 / 1e6:.2f} MB')

## Phase 6 — Training

In [ ]:
EPOCHS   = 30
LR       = 3e-4
LR_MIN   = 1e-6

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=LR_MIN
)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0

def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for x, y in tqdm(loader, leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss   = criterion(logits, y)

            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item() * x.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total   += x.size(0)

    return total_loss / total, correct / total


print('Starting training...\n')
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(train_dl, training=True)
    vl_loss, vl_acc = run_epoch(val_dl,   training=False)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    elapsed = time.time() - t0
    print(f'Epoch {epoch:02d}/{EPOCHS} | '
          f'Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | '
          f'Val Loss: {vl_loss:.4f} Acc: {vl_acc:.4f} | '
          f'Time: {elapsed:.1f}s')

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), 'best_sonar_model.pt')
        print(f'  ✓ Saved best model (val_acc={best_val_acc:.4f})')

print(f'\nBest Validation Accuracy: {best_val_acc:.4f}')

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], label='Train Loss', color='royalblue')
ax1.plot(history['val_loss'],   label='Val Loss',   color='tomato')
ax1.set_title('Loss Curves', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(history['train_acc'], label='Train Acc', color='royalblue')
ax2.plot(history['val_acc'],   label='Val Acc',   color='tomato')
ax2.set_title('Accuracy Curves', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## Phase 7 — INT8 Quantization + Dynamic Scale Comparison

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load('best_sonar_model.pt', map_location=DEVICE))
model.eval()

# ── INT8 Dynamic Quantization ─────────────────────────────────────────────────
# Quantize all Linear layers to INT8 (weights only — matches FPGA INT8 MAC units)
model_fp32 = model.cpu()
model_int8 = quantize_dynamic(
    model_fp32,
    {nn.Linear},
    dtype=torch.qint8
)

def get_model_size_mb(m):
    torch.save(m.state_dict(), '_tmp.pt')
    size = os.path.getsize('_tmp.pt') / 1e6
    os.remove('_tmp.pt')
    return size

fp32_size = get_model_size_mb(model_fp32)
int8_size = get_model_size_mb(model_int8)

print(f'FP32 model size : {fp32_size:.2f} MB')
print(f'INT8 model size : {int8_size:.2f} MB')
print(f'Compression     : {fp32_size/int8_size:.2f}x')

In [ ]:
# ── Evaluate Both Models on Test Set ─────────────────────────────────────────
def evaluate(m, loader, label='Model'):
    m.eval()
    correct, total = 0, 0
    all_preds, all_labels = [], []
    t0 = time.time()

    with torch.no_grad():
        for x, y in loader:
            logits = m(x)          # CPU inference for quantized model
            preds  = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total   += y.size(0)
            all_preds.extend(preds.numpy())
            all_labels.extend(y.numpy())

    elapsed = time.time() - t0
    acc = correct / total
    print(f'\n── {label} ──')
    print(f'Accuracy    : {acc:.4f}')
    print(f'Latency     : {elapsed:.2f}s total | {elapsed/total*1000:.2f} ms/sample')
    print(classification_report(all_labels, all_preds, target_names=CLASSES))
    return acc, all_preds, all_labels

# Use CPU DataLoader for quantized model comparison
test_dl_cpu = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

acc_fp32, preds_fp32, labels = evaluate(model_fp32, test_dl_cpu, 'FP32 (Baseline)')
acc_int8, preds_int8, _      = evaluate(model_int8, test_dl_cpu, 'INT8 Quantized (Dynamic Scale)')

In [ ]:
# ── Jamming Attack Simulation ─────────────────────────────────────────────────
# Inject Gaussian noise to simulate ocean acoustic jamming
# This is the key demo: standard INT8 degrades, dynamic-scale INT8 holds.

# Build baseline model WITHOUT dynamic scale for comparison
model_nodyn = SonarTransformer(use_dynamic_scale=False)
model_nodyn.load_state_dict(torch.load('best_sonar_model.pt', map_location='cpu'),
                             strict=False)
model_nodyn_int8 = quantize_dynamic(model_nodyn, {nn.Linear}, dtype=torch.qint8)

noise_levels   = [0.0, 0.1, 0.3, 0.5, 0.8, 1.2, 2.0]
acc_dynamic    = []
acc_static     = []

for noise_std in noise_levels:
    c_dyn, c_sta, total = 0, 0, 0
    with torch.no_grad():
        for x, y in test_dl_cpu:
            x_noisy = x + torch.randn_like(x) * noise_std
            p_dyn   = model_int8(x_noisy).argmax(dim=1)
            p_sta   = model_nodyn_int8(x_noisy).argmax(dim=1)
            c_dyn  += (p_dyn == y).sum().item()
            c_sta  += (p_sta == y).sum().item()
            total  += y.size(0)
    acc_dynamic.append(c_dyn / total)
    acc_static.append(c_sta / total)
    print(f'Noise σ={noise_std:.1f} | Dynamic: {c_dyn/total:.3f} | Static: {c_sta/total:.3f}')

# Plot
plt.figure(figsize=(10, 5))
plt.plot(noise_levels, acc_dynamic, 'o-', color='royalblue',
         label='INT8 + Dynamic Scale (Ours)', linewidth=2.5)
plt.plot(noise_levels, acc_static,  's--', color='tomato',
         label='INT8 Static (Baseline)', linewidth=2.5)
plt.xlabel('Jamming Noise σ', fontsize=13)
plt.ylabel('Classification Accuracy', fontsize=13)
plt.title('Robustness Under Acoustic Jamming — Dynamic vs Static INT8',
          fontsize=13, fontweight='bold')
plt.legend(fontsize=11); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('jamming_robustness.png', dpi=150)
plt.show()
print('Saved: jamming_robustness.png')

## Phase 8 — FPGA Resource Estimation
### Xilinx Zynq UltraScale+ Simulation

In [ ]:
# ── FPGA Resource Estimator ───────────────────────────────────────────────────
# Reference: Xilinx Zynq UltraScale+ ZU9EG
# Resources:  2,520 DSP48E2 | 32.1 Mb BRAM | 600,000 LUTs
# Assumes INT8 MACs → each DSP48 handles 1 INT8 MAC/cycle @ 200MHz

FPGA_SPECS = {
    'name'       : 'Zynq UltraScale+ ZU9EG',
    'DSP48'      : 2520,
    'BRAM_Mb'    : 32.1,
    'LUT'        : 600000,
    'freq_MHz'   : 200,
}

CONFIG = {
    'embed_dim'  : 256,
    'num_heads'  : 8,
    'depth'      : 6,
    'mlp_ratio'  : 4.0,
    'num_patches': NUM_PATCHES,       # 104
    'seq_len'    : NUM_PATCHES + 1,   # +CLS
}

D  = CONFIG['embed_dim']
H  = CONFIG['num_heads']
N  = CONFIG['seq_len']
Dh = D // H
L  = CONFIG['depth']

# MACs per layer type (one forward pass)
mac_qkv        = L * N * D * (3 * D)            # QKV projection
mac_attn       = L * H * N * N * Dh             # QKᵀ
mac_attn_v     = L * H * N * N * Dh             # Attn × V
mac_out        = L * N * D * D                  # output projection
mac_mlp        = L * N * (D * int(D*4) + int(D*4) * D)  # FFN
mac_classifier = D * (D // 2) + (D // 2) * len(CLASSES)

total_macs = (mac_qkv + mac_attn + mac_attn_v +
              mac_out + mac_mlp + mac_classifier)

# BRAM: weights + KV cache at INT8 (1 byte/element)
weight_bytes  = total_params              # INT8 quantized weights
kv_cache_bytes = L * H * N * Dh * 2     # K + V cache
total_bytes   = weight_bytes + kv_cache_bytes

# DSP estimate: use ~30% of available for this core (leave room for control)
dsp_used_est  = min(int(total_macs / (FPGA_SPECS['freq_MHz'] * 1e6) * 1000), 
                    int(FPGA_SPECS['DSP48'] * 0.75))

# Latency estimate @ 200MHz, assuming full pipeline utilization
cycles_total    = total_macs / dsp_used_est if dsp_used_est > 0 else total_macs
latency_ms      = cycles_total / (FPGA_SPECS['freq_MHz'] * 1e6) * 1000

print('=' * 60)
print(f'  FPGA Resource Estimation — {FPGA_SPECS["name"]}')
print('=' * 60)
print(f'  Total MACs (1 inference) : {total_macs/1e9:.3f} GMACs')
print(f'  Weight storage (INT8)    : {weight_bytes/1e6:.2f} MB')
print(f'  KV Cache (INT8)          : {kv_cache_bytes/1e3:.2f} KB')
print(f'  Total on-chip memory     : {total_bytes/1e6:.2f} MB')
print(f'  BRAM available           : {FPGA_SPECS["BRAM_Mb"]:.1f} Mb = {FPGA_SPECS["BRAM_Mb"]/8:.2f} MB')
print(f'  BRAM utilization         : {total_bytes/1e6 / (FPGA_SPECS["BRAM_Mb"]/8) * 100:.1f}%')
print(f'  DSP48 used (est.)        : {dsp_used_est} / {FPGA_SPECS["DSP48"]}')
print(f'  DSP utilization          : {dsp_used_est/FPGA_SPECS["DSP48"]*100:.1f}%')
print(f'  Estimated latency        : {latency_ms:.2f} ms/inference')
print(f'  Throughput               : {1000/latency_ms:.1f} inferences/sec')
print('=' * 60)

if total_bytes / 1e6 > FPGA_SPECS['BRAM_Mb'] / 8:
    print('⚠  Exceeds BRAM — DDR4 offload needed for weights')
    print('   Mitigation: reduce depth to 4 or embed_dim to 128')
else:
    print('✓  Fits on-chip BRAM — no DRAM dependency')

In [ ]:
# ── FPGA Resource Visualization ───────────────────────────────────────────────
resources = {
    'DSP48E2': dsp_used_est / FPGA_SPECS['DSP48'] * 100,
    'BRAM'   : total_bytes / 1e6 / (FPGA_SPECS['BRAM_Mb'] / 8) * 100,
    'LUT (est.)': 35.0,   # Conservative estimate for control logic
}

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(list(resources.keys()), list(resources.values()),
               color=['royalblue', 'seagreen', 'darkorange'], height=0.5)
ax.axvline(x=80, color='red', linestyle='--', linewidth=1.5, label='80% limit')
ax.axvline(x=100, color='darkred', linestyle='-', linewidth=1.5, label='100% (max)')
ax.set_xlabel('Utilization (%)', fontsize=12)
ax.set_title(f'FPGA Resource Utilization — {FPGA_SPECS["name"]}',
             fontsize=12, fontweight='bold')
for bar, val in zip(bars, resources.values()):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=11)
ax.legend(); ax.set_xlim(0, 115); ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('fpga_resources.png', dpi=150)
plt.show()

## Phase 9 — Attention Map Visualization
### What is the model attending to in sonar signals?

In [ ]:
model_fp32.eval()
sample_x, sample_y = next(iter(test_dl_cpu))
sample_x = sample_x[:4]  # 4 samples

with torch.no_grad():
    logits, attn_maps = model_fp32(sample_x, return_attn=True)
    preds = logits.argmax(dim=1)

# Use last layer attention, average over heads, CLS token attention
last_attn = attn_maps[-1]              # (B, H, N+1, N+1)
cls_attn  = last_attn[:, :, 0, 1:]    # (B, H, N) — CLS attending to patches
cls_attn  = cls_attn.mean(dim=1)      # (B, N) — average over heads

pH = IMG_H // PATCH_SIZE  # 8
pW = IMG_W // PATCH_SIZE  # 13

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle('Attention Maps — What the Model Focuses On in Sonar',
             fontsize=13, fontweight='bold')

for i in range(4):
    mel_img = sample_x[i, 0].numpy()
    attn    = cls_attn[i].reshape(pH, pW).numpy()
    attn    = (attn - attn.min()) / (attn.max() - attn.min() + 1e-8)

    # Upsample attention map to spectrogram size
    import torch.nn.functional as F_
    attn_up = F_.interpolate(
        torch.tensor(attn).unsqueeze(0).unsqueeze(0),
        size=(IMG_H, IMG_W), mode='bilinear', align_corners=False
    ).squeeze().numpy()

    axes[0][i].imshow(mel_img, aspect='auto', origin='lower', cmap='viridis')
    axes[0][i].set_title(f'Input: {CLASSES[sample_y[i]]}\nPred: {CLASSES[preds[i]]}',
                          fontsize=10)
    axes[0][i].axis('off')

    axes[1][i].imshow(mel_img, aspect='auto', origin='lower', cmap='viridis', alpha=0.6)
    axes[1][i].imshow(attn_up, aspect='auto', origin='lower', cmap='hot', alpha=0.5)
    axes[1][i].set_title('Attention Overlay', fontsize=10)
    axes[1][i].axis('off')

plt.tight_layout()
plt.savefig('attention_maps.png', dpi=150)
plt.show()

## Phase 10 — Final Results Summary

In [ ]:
print('=' * 65)
print('  FINAL RESULTS — Passive Sonar Threat Classification')
print('=' * 65)
print(f'  Architecture    : Quantized Vision Transformer (ViT-Small)')
print(f'  Patch size      : {PATCH_SIZE}×{PATCH_SIZE} | Tokens: {NUM_PATCHES}')
print(f'  Embed dim       : {D} | Heads: {H} | Depth: {L}')
print(f'  Total params    : {total_params:,}')
print()
print(f'  FP32 Test Acc   : {acc_fp32:.4f}')
print(f'  INT8 Test Acc   : {acc_int8:.4f}')
print(f'  Accuracy drop   : {(acc_fp32 - acc_int8)*100:.2f}%')
print(f'  Size reduction  : {fp32_size:.2f} MB → {int8_size:.2f} MB ({fp32_size/int8_size:.1f}x)')
print()
print(f'  FPGA Target     : {FPGA_SPECS["name"]}')
print(f'  Est. Latency    : {latency_ms:.2f} ms/inference')
print(f'  DSP Utilization : {dsp_used_est}/{FPGA_SPECS["DSP48"]} ({dsp_used_est/FPGA_SPECS["DSP48"]*100:.1f}%)')
print(f'  BRAM Usage      : {total_bytes/1e6:.2f} MB')
print()
print('  Saved artifacts :')
print('    sonar_spectrograms.png')
print('    training_curves.png')
print('    jamming_robustness.png')
print('    fpga_resources.png')
print('    attention_maps.png')
print('    best_sonar_model.pt')
print('=' * 65)